
<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Imagenes/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Propedéutico en Programación y Algoritmos para la Maestría en Ciencia de Datos</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 3: Manipulación de Datos con la librería Pandas II</h3>
    </div>
</div>


<img style="float: right; margin: 0px 0px 15px 15px;" src="https://numfocus.org/wp-content/uploads/2016/07/pandas-logo-300.png" width="400px" height="400px" />

> La clase pasada comenzamos a ver las funcionalidades de la librería [pandas](https://pandas.pydata.org/). Básicamente, aprendimos a cargar datos desde archivos `.csv`, aprendimos a seleccionar subcojuntos de estos datos mediante indización (obtener ciertas filas, o ciertas columnas, o ciertas filas y columnas determinadas). Finalmente, aprendimos a filtrar datos, es decir, a seleccionar registros de la base de datos que satisfagan cierta condición.

> Hoy nos enfocaremos a ver como reunir varias bases de datos que poseen información complementaria relativa a un mismo problema, para obtener una sola tabla con la información condensada.

Referencias:
- https://www.shanelynn.ie/merge-join-dataframes-python-pandas-index-1/
___

# 0. Motivación

En cualquier situación relacionada con ciencia de datos en la vida real, no pasarán más de 10 minutos sin que aparezca la necesidad de unir (`merge` o `join`) dos bases de datos complementarias para formar tu conjunto de datos para el análisis.

La unión de DataFrames es un proceso fundamental, que cualquier analista de datos en formación debe aprender. En esta clase aprenderemos cómo hacerlo, y en el camino responderemos las siguientes preguntas:

- ¿Qué son el `merge` o `join` de dos DataFrames?

- ¿Qué son `inner`, `outer`, `left` y `right` `joins`?



## Datos para la clase

En esta clase, seguiremos trabajando con los datos de la clase pasada:

En la carpeta "data" tenemos los archivos 

- "customers_data.csv": datos propios de los clientes de la empresa, 
- "sessions_data.csv": inicios de sesión de los clientes en la plataforma web de compras (similar a amazon),
- "transactions_data.csv": transacciones asociadas a cada inicio de sesión, y
- "products_data.csv": información de los productos. 

Podemos cargar cada uno de los archivos `.csv` como DataFrames de Pandas usando la función `read_csv()` de pandas, y examinar los contenidos de cada uno con el método `head()`.

In [1]:
# Importar pandas
import pandas as pd

In [2]:
# Cargar customers_data
customers = pd.read_csv('Data/customers_data.csv', index_col=[0])
customers

,customer_id,zip_code,join_date,date_of_birth
0,1,60091,2011-04-17 10:48:33,1994-07-18
1,2,13244,2012-04-15 23:31:04,1986-08-18
2,3,13244,2011-08-13 15:42:34,2003-11-21
3,4,60091,2011-04-08 20:08:14,2006-08-15
4,5,60091,2010-07-17 05:27:50,1984-07-28


In [3]:
customers.shape

(5, 4)

In [4]:
# Cargar sessions_data
sesions = pd.read_csv('Data/sessions_data.csv', index_col=[0])
sesions

,session_id,customer_id,device,session_start
0,1,2,desktop,2014-01-01 00:00:00
1,2,5,mobile,2014-01-01 00:17:20
2,3,4,mobile,2014-01-01 00:28:10
3,4,1,mobile,2014-01-01 00:44:25
4,5,4,mobile,2014-01-01 01:11:30
5,6,1,tablet,2014-01-01 01:23:25
6,7,3,tablet,2014-01-01 01:39:40
7,8,4,tablet,2014-01-01 01:55:55
8,9,1,desktop,2014-01-01 02:15:25
9,10,2,tablet,2014-01-01 02:31:40


In [13]:
sesions.head()

,session_id,customer_id,device,session_start
0,1,2,desktop,2014-01-01 00:00:00
1,2,5,mobile,2014-01-01 00:17:20
2,3,4,mobile,2014-01-01 00:28:10
3,4,1,mobile,2014-01-01 00:44:25
4,5,4,mobile,2014-01-01 01:11:30


In [ ]:
sesions['device'].unique() #regresa serie

<StringArray>
['desktop', 'mobile', 'tablet']
Length: 3, dtype: str

In [5]:
# Cargar transactions_data
transactions = pd.read_csv('Data/transactions_data.csv', index_col=[0])
transactions

,transaction_id,session_id,transaction_time,product_id,amount
0,298,1,2014-01-01 00:00:00,5,127.64
1,2,1,2014-01-01 00:01:05,2,109.48
2,308,1,2014-01-01 00:02:10,3,95.06
3,116,1,2014-01-01 00:03:15,4,78.92
4,371,1,2014-01-01 00:04:20,3,31.54
...,...,...,...,...,...
495,112,35,2014-01-01 08:56:15,5,55.42
496,111,35,2014-01-01 08:57:20,3,34.87
497,276,35,2014-01-01 08:58:25,1,10.94
498,266,35,2014-01-01 08:59:30,5,19.86


In [15]:
transactions.session_id

0       1
1       1
2       1
3       1
4       1
       ..
495    35
496    35
497    35
498    35
499    35
Name: session_id, Length: 500, dtype: int64

In [ ]:
transactions.iloc[0,2] #regresa cadena de texto

'2014-01-01 00:00:00'

In [6]:
# Cargar products_data
products = pd.read_csv('Data/products_data.csv', index_col=[0])
products

,product_id,brand
0,1,B
1,2,B
2,3,B
3,4,B
4,5,A


## Bonus: formato de fechas y horas

Como vimos la vez pasada, es relevante en general un correcto manejo de las fechas y horas en las bases de datos.

Lo primero que se tiene que hacer para un correcto manejo de las fechas (y horas) es identificar las columnas o variables que contienen fechas. Por ejemplo, de la tabla `customers_data`:

Lo que sigue, es especificarle a pandas el formato en que vienen esas fechas con la función `to_datetime()` de pandas

In [7]:
# Cargar customers_data
customers = pd.read_csv('Data/customers_data.csv', index_col=[0])
customers

,customer_id,zip_code,join_date,date_of_birth
0,1,60091,2011-04-17 10:48:33,1994-07-18
1,2,13244,2012-04-15 23:31:04,1986-08-18
2,3,13244,2011-08-13 15:42:34,2003-11-21
3,4,60091,2011-04-08 20:08:14,2006-08-15
4,5,60091,2010-07-17 05:27:50,1984-07-28


In [18]:
# Ayuda en la función to_datetime
help(pd.to_datetime)

Help on function to_datetime in module pandas:

to_datetime(
    arg: DatetimeScalarOrArrayConvertible | DictConvertible,
    errors: DateTimeErrorChoices = 'raise',
    dayfirst: bool = False,
    yearfirst: bool = False,
    utc: bool = False,
    format: str | None = None,
    exact: bool | lib.NoDefault = <no_default>,
    unit: str | None = None,
    origin: str = 'unix',
    cache: bool = True
) -> DatetimeIndex | Series | DatetimeScalar | NaTType
    Convert argument to datetime.

    This function converts a scalar, array-like, :class:`Series` or
    :class:`DataFrame`/dict-like to a pandas datetime object.

    Parameters
    ----------
    arg : int, float, str, datetime, list, tuple, 1-d array, Series, DataFrame/dict-like
        The object to convert to a datetime. If a :class:`DataFrame` is provided, the
        method expects minimally the following columns: :const:`"year"`,
        :const:`"month"`, :const:`"day"`. The column "year"
        must be specified in 4-digit f

In [ ]:
# Especificar el formato de fechas en la tabla customers_data


In [19]:
customers['join_date'] = pd.to_datetime(customers['join_date'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
customers['join_date']

0   2011-04-17 10:48:33
1   2012-04-15 23:31:04
2   2011-08-13 15:42:34
3   2011-04-08 20:08:14
4   2010-07-17 05:27:50
Name: join_date, dtype: datetime64[us]

In [9]:
customers

,customer_id,zip_code,join_date,date_of_birth
0,1,60091,2011-04-17 10:48:33,1994-07-18
1,2,13244,2012-04-15 23:31:04,1986-08-18
2,3,13244,2011-08-13 15:42:34,2003-11-21
3,4,60091,2011-04-08 20:08:14,2006-08-15
4,5,60091,2010-07-17 05:27:50,1984-07-28


In [10]:
customers['date_of_birth']=pd.to_datetime(customers['date_of_birth'], format='%Y-%m-%d', errors = 'coerce')
customers['date_of_birth']

0   1994-07-18
1   1986-08-18
2   2003-11-21
3   2006-08-15
4   1984-07-28
Name: date_of_birth, dtype: datetime64[us]

In [20]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    5 non-null      int64         
 1   zip_code       5 non-null      int64         
 2   join_date      5 non-null      datetime64[us]
 3   date_of_birth  5 non-null      datetime64[us]
dtypes: datetime64[us](2), int64(2)
memory usage: 292.0 bytes


Hagan esto mismo para todas las tablas:

In [11]:
sesions.head()

,session_id,customer_id,device,session_start
0,1,2,desktop,2014-01-01 00:00:00
1,2,5,mobile,2014-01-01 00:17:20
2,3,4,mobile,2014-01-01 00:28:10
3,4,1,mobile,2014-01-01 00:44:25
4,5,4,mobile,2014-01-01 01:11:30


In [23]:
# Especificar el formato de fechas en la tabla customers
transactions['transaction_time'] = pd.to_datetime(transactions['transaction_time'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
transactions['transaction_time']

0     2014-01-01 00:00:00
1     2014-01-01 00:01:05
2     2014-01-01 00:02:10
3     2014-01-01 00:03:15
4     2014-01-01 00:04:20
              ...        
495   2014-01-01 08:56:15
496   2014-01-01 08:57:20
497   2014-01-01 08:58:25
498   2014-01-01 08:59:30
499   2014-01-01 09:00:35
Name: transaction_time, Length: 500, dtype: datetime64[us]

## Volviendo al problema ...

Hay columnas o variables que relacionan las tablas. Por ejemplo:

- Los clientes pueden iniciar sesión en la plataforma cuantas veces quieran. De manera que hay una relación uno a muchos entre las tablas "customers_data" y "sessions_data", mediante la variable "customer_id".

- Cuando se efectúa una transacción se supone que se está comprando un producto. Por lo tanto hay una relación uno a muchos entre las tablas "transactions_data"  y "products_data", mediante la variable "product_id".

## Problemas

Primero, quisieramos determinar cuál es el dispositivo preferido por zona.

Luego, quisiéramos determinar cuál es la marca preferida por zona.

Para resolver lo anterior necesitamos unir ("merge" o "join") nuestros datasets en uno solo para el análisis.

___
# 1. Uniendo dos DataFrames...

> “Merging” two datasets is the process of bringing two datasets together into one, and aligning the rows from each based on common attributes or columns.

Las palabras “merge” y “join” se usan indistintamente en Pandas, y en otros lenguajes como SQL y R. En Pandas, hay métodos separados “merge” y “join”, que realizan cosas similares (personalmente uso el método "merge"), y la función "merge".

Vamos a concentrarnos en el **primer problema**. En este escenario, necesitamos llevar a cabo los siguientes pasos:

- Para cada fila en el dataset `sessions_data`, debemos hacer una nueva columna que contenga el "zip_code" respectivo de cada cliente.
- Una vez hagamos esto, obtenemos la moda de los dispositivos para cada cliente.

**¿Podemos usar un ciclo?**

Claro que si. Se podría escribir un ciclo para esta tarea. Éste iría a través de cada fila en `sessions_data`, y a cada "user_id" asignar el valor de la nueva columna con la zona respectiva.

Sin embargo, usar ciclos haría nuestra tarea mucho más lenta y plagada de más código que el necesario que si se usara la función (método) `join()`.

De forma que, para estas situaciones, **nunca usar ciclos**.

Para ver cómo podemos hacer lo anterior, veamos la ayuda de la función `merge()` de pandas:

In [24]:
# Ayuda de merge()
help(pd.merge)

Help on function merge in module pandas:

merge(
    left: DataFrame | Series,
    right: DataFrame | Series,
    how: MergeHow = 'inner',
    on: IndexLabel | AnyArrayLike | None = None,
    left_on: IndexLabel | AnyArrayLike | None = None,
    right_on: IndexLabel | AnyArrayLike | None = None,
    left_index: bool = False,
    right_index: bool = False,
    sort: bool = False,
    suffixes: Suffixes = ('_x', '_y'),
    copy: bool | lib.NoDefault = <no_default>,
    indicator: str | bool = False,
    validate: str | None = None
) -> DataFrame
    Merge DataFrame or named Series objects with a database-style join.

    A named Series object is treated as a DataFrame with a single named column.

    The join is done on columns or indexes. If joining columns on
    columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
    on indexes or indexes on a column or columns, the index will be passed on.
    When performing a cross merge, no column specifications to merge

Ahora, veamos como podemos añadir el código postal a la tabla `sessions_data`, usando la función `merge()` de pandas:

In [27]:
# Uso de la función merge()
# sessions_with_zip = pd.merge(left=sessions,
#                              right=customers[['customer_id', 'zip_code']]
#                              on="customer_id")
sesions_with_zip = pd.merge(left=sesions, right=customers[['customer_id', 'zip_code']], on = 'customer_id', how = 'inner')
sesions_with_zip

,session_id,customer_id,device,session_start,zip_code
0,1,2,desktop,2014-01-01 00:00:00,13244
1,2,5,mobile,2014-01-01 00:17:20,60091
2,3,4,mobile,2014-01-01 00:28:10,60091
3,4,1,mobile,2014-01-01 00:44:25,60091
4,5,4,mobile,2014-01-01 01:11:30,60091
5,6,1,tablet,2014-01-01 01:23:25,60091
6,7,3,tablet,2014-01-01 01:39:40,13244
7,8,4,tablet,2014-01-01 01:55:55,60091
8,9,1,desktop,2014-01-01 02:15:25,60091
9,10,2,tablet,2014-01-01 02:31:40,13244


In [25]:
sesions.columns

Index(['session_id', 'customer_id', 'device', 'session_start'], dtype='str')

La función `merge()` es el objetivo principal de esta clase. Básicamente, la operación de unir dos DataFrames hace lo siguiente: 

- toma el DataFrame de la izquierda (argumento left=), 
- el DataFrame de la derecha (argumento right=),
- la columna donde se va a unir (argumento on=), y
- la forma en que se va a unir (argumento how=).

La variable en común sobre la que se hace la unión está especificada en el argumento `on`.

Con este resultado, podemos filtrar por zona y luego obtener la moda.

In [28]:
# Obtener la moda por zona
sesions_with_zip.head()

,session_id,customer_id,device,session_start,zip_code
0,1,2,desktop,2014-01-01 00:00:00,13244
1,2,5,mobile,2014-01-01 00:17:20,60091
2,3,4,mobile,2014-01-01 00:28:10,60091
3,4,1,mobile,2014-01-01 00:44:25,60091
4,5,4,mobile,2014-01-01 01:11:30,60091


In [30]:
sesions_with_zip['zip_code'].unique()

array([13244, 60091])

In [33]:
sesions_with_zip[sesions_with_zip['zip_code'] == 13244]

,session_id,customer_id,device,session_start,zip_code
0,1,2,desktop,2014-01-01 00:00:00,13244
6,7,3,tablet,2014-01-01 01:39:40,13244
9,10,2,tablet,2014-01-01 02:31:40,13244
14,15,2,desktop,2014-01-01 03:41:00,13244
15,16,2,desktop,2014-01-01 03:49:40,13244
16,17,2,tablet,2014-01-01 04:00:30,13244
18,19,3,desktop,2014-01-01 04:27:35,13244
22,23,3,desktop,2014-01-01 05:32:35,13244
24,25,3,desktop,2014-01-01 05:59:40,13244
30,31,2,mobile,2014-01-01 07:42:35,13244


In [34]:
sesions_with_zip['zip_code'].unique

<bound method Series.unique of 0     13244
1     60091
2     60091
3     60091
4     60091
5     60091
6     13244
7     60091
8     60091
9     13244
10    60091
11    60091
12    60091
13    60091
14    13244
15    13244
16    13244
17    60091
18    13244
19    60091
20    60091
21    60091
22    13244
23    60091
24    13244
25    60091
26    60091
27    60091
28    60091
29    60091
30    13244
31    60091
32    13244
33    13244
34    13244
Name: zip_code, dtype: int64>

In [36]:
s_zipcode1 = sesions_with_zip[sesions_with_zip['zip_code'] == 13244]
s_zipcode1

,session_id,customer_id,device,session_start,zip_code
0,1,2,desktop,2014-01-01 00:00:00,13244
6,7,3,tablet,2014-01-01 01:39:40,13244
9,10,2,tablet,2014-01-01 02:31:40,13244
14,15,2,desktop,2014-01-01 03:41:00,13244
15,16,2,desktop,2014-01-01 03:49:40,13244
16,17,2,tablet,2014-01-01 04:00:30,13244
18,19,3,desktop,2014-01-01 04:27:35,13244
22,23,3,desktop,2014-01-01 05:32:35,13244
24,25,3,desktop,2014-01-01 05:59:40,13244
30,31,2,mobile,2014-01-01 07:42:35,13244


In [38]:
sesions_with_zip.loc[sesions_with_zip['zip_code'] == 13244, 'device'].value_counts()

device
desktop    7
tablet     3
mobile     3
Name: count, dtype: int64

In [39]:
sesions_with_zip.loc[sesions_with_zip['zip_code'] == 60091, 'device'].value_counts()

device
mobile     10
desktop     7
tablet      5
Name: count, dtype: int64

In [45]:
for zipcode in sesions_with_zip['zip_code'].unique():
    print(sesions_with_zip.loc[sesions_with_zip['zip_code'] == zipcode, 'device'].value_counts())

device
desktop    7
tablet     3
mobile     3
Name: count, dtype: int64
device
mobile     10
desktop     7
tablet      5
Name: count, dtype: int64


### Usando GroupBy

In [46]:
s_groupby = sesions_with_zip.groupby(by='zip_code')['device'].value_counts()

In [47]:
s_groupby

zip_code  device 
13244     desktop     7
          tablet      3
          mobile      3
60091     mobile     10
          desktop     7
          tablet      5
Name: count, dtype: int64

In [48]:
type(s_groupby)

pandas.Series

### ¿Qué son los tipos de unión inner, left, right y outer?

En nuestro ejemplo, unimos `sessions` con `customers` sobre la columna "custumers_id". En este caso, en ambas tablas existían todos los posibles valores de "customers_id" (1, 2, 3, 4, 5).

¿Qué pasa si esta situación no se cumple?

Por ejemplo, realicemos los siguientes cambios artificiales:

In [50]:
sesions['customer_id'].unique()

array([2, 5, 4, 1, 3])

In [53]:
s_artificial = sesions[sesions['customer_id'] != 2]
s_artificial

,session_id,customer_id,device,session_start
1,2,5,mobile,2014-01-01 00:17:20
2,3,4,mobile,2014-01-01 00:28:10
3,4,1,mobile,2014-01-01 00:44:25
4,5,4,mobile,2014-01-01 01:11:30
5,6,1,tablet,2014-01-01 01:23:25
6,7,3,tablet,2014-01-01 01:39:40
7,8,4,tablet,2014-01-01 01:55:55
8,9,1,desktop,2014-01-01 02:15:25
10,11,4,mobile,2014-01-01 02:47:55
11,12,4,desktop,2014-01-01 03:04:10


In [56]:
s_artificial['customer_id'].unique()

array([5, 4, 1, 3])

In [58]:
c_art = customers[customers['customer_id'] != 5]
c_art

,customer_id,zip_code,join_date,date_of_birth
0,1,60091,2011-04-17 10:48:33,1994-07-18
1,2,13244,2012-04-15 23:31:04,1986-08-18
2,3,13244,2011-08-13 15:42:34,2003-11-21
3,4,60091,2011-04-08 20:08:14,2006-08-15


Por defecto, la operación `merge()` de pandas actpua con un merge tipo "inner". Un "inner merge", guarda únicamente los valores comúnes (en la columna especificada en el argumento `on=`) de ambos DataFrames.

Por ejemplo:

Los otros comportamientos se pueden revisar mejor a través de ejemplos:

In [60]:
# left join
sesions_with_zip_art = pd.merge(left=s_artificial, right=c_art, on = 'customer_id', how = 'left')
sesions_with_zip_art.head()


,session_id,customer_id,device,session_start,zip_code,join_date,date_of_birth
0,2,5,mobile,2014-01-01 00:17:20,NaN,NaT,NaT
1,3,4,mobile,2014-01-01 00:28:10,60091.0,2011-04-08 20:08:14,2006-08-15
2,4,1,mobile,2014-01-01 00:44:25,60091.0,2011-04-17 10:48:33,1994-07-18
3,5,4,mobile,2014-01-01 01:11:30,60091.0,2011-04-08 20:08:14,2006-08-15
4,6,1,tablet,2014-01-01 01:23:25,60091.0,2011-04-17 10:48:33,1994-07-18


In [61]:
sesions_with_zip_art['customer_id'].unique()

array([5, 4, 1, 3])

In [68]:
# right join
sesions_with_zip_art = pd.merge(left=s_artificial, right=c_art, on = 'customer_id', how = 'right')
sesions_with_zip_art.head()


,session_id,customer_id,device,session_start,zip_code,join_date,date_of_birth
0,4.0,1,mobile,2014-01-01 00:44:25,60091,2011-04-17 10:48:33,1994-07-18
1,6.0,1,tablet,2014-01-01 01:23:25,60091,2011-04-17 10:48:33,1994-07-18
2,9.0,1,desktop,2014-01-01 02:15:25,60091,2011-04-17 10:48:33,1994-07-18
3,14.0,1,tablet,2014-01-01 03:28:00,60091,2011-04-17 10:48:33,1994-07-18
4,18.0,1,desktop,2014-01-01 04:14:35,60091,2011-04-17 10:48:33,1994-07-18


In [69]:
sesions_with_zip_art['customer_id'].unique()

array([1, 2, 3, 4])

In [70]:
# outer join
sesions_with_zip_art = pd.merge(left=s_artificial, right=c_art, on = 'customer_id', how = 'outer')
sesions_with_zip_art.head()


,session_id,customer_id,device,session_start,zip_code,join_date,date_of_birth
0,4.0,1,mobile,2014-01-01 00:44:25,60091.0,2011-04-17 10:48:33,1994-07-18
1,6.0,1,tablet,2014-01-01 01:23:25,60091.0,2011-04-17 10:48:33,1994-07-18
2,9.0,1,desktop,2014-01-01 02:15:25,60091.0,2011-04-17 10:48:33,1994-07-18
3,14.0,1,tablet,2014-01-01 03:28:00,60091.0,2011-04-17 10:48:33,1994-07-18
4,18.0,1,desktop,2014-01-01 04:14:35,60091.0,2011-04-17 10:48:33,1994-07-18


In [71]:
sesions_with_zip_art['customer_id'].unique()

array([1, 2, 3, 4, 5])

In [72]:
sesions_with_zip_art

,session_id,customer_id,device,session_start,zip_code,join_date,date_of_birth
0,4.0,1,mobile,2014-01-01 00:44:25,60091.0,2011-04-17 10:48:33,1994-07-18
1,6.0,1,tablet,2014-01-01 01:23:25,60091.0,2011-04-17 10:48:33,1994-07-18
2,9.0,1,desktop,2014-01-01 02:15:25,60091.0,2011-04-17 10:48:33,1994-07-18
3,14.0,1,tablet,2014-01-01 03:28:00,60091.0,2011-04-17 10:48:33,1994-07-18
4,18.0,1,desktop,2014-01-01 04:14:35,60091.0,2011-04-17 10:48:33,1994-07-18
5,26.0,1,tablet,2014-01-01 06:17:00,60091.0,2011-04-17 10:48:33,1994-07-18
6,27.0,1,mobile,2014-01-01 06:34:20,60091.0,2011-04-17 10:48:33,1994-07-18
7,29.0,1,mobile,2014-01-01 07:10:05,60091.0,2011-04-17 10:48:33,1994-07-18
8,NaN,2,NaN,NaT,13244.0,2012-04-15 23:31:04,1986-08-18
9,7.0,3,tablet,2014-01-01 01:39:40,13244.0,2011-08-13 15:42:34,2003-11-21


Estos comportamientos son muy intuitivos a partir de sus valores.

Así mismo, sus usos son muy intuitivos (lo sabrán cuando se enfrenten a problemas reales).

### Resolvamos el problema 2

Determinar cuál es la marca preferida por zona.

In [73]:
sesions_with_zip.head()

,session_id,customer_id,device,session_start,zip_code
0,1,2,desktop,2014-01-01 00:00:00,13244
1,2,5,mobile,2014-01-01 00:17:20,60091
2,3,4,mobile,2014-01-01 00:28:10,60091
3,4,1,mobile,2014-01-01 00:44:25,60091
4,5,4,mobile,2014-01-01 01:11:30,60091


In [74]:
products

,product_id,brand
0,1,B
1,2,B
2,3,B
3,4,B
4,5,A


In [75]:
transactions_products = transactions.merge(right=products, on = 'product_id', how = 'inner')
transactions_products

,transaction_id,session_id,transaction_time,product_id,amount,brand
0,298,1,2014-01-01 00:00:00,5,127.64,A
1,2,1,2014-01-01 00:01:05,2,109.48,B
2,308,1,2014-01-01 00:02:10,3,95.06,B
3,116,1,2014-01-01 00:03:15,4,78.92,B
4,371,1,2014-01-01 00:04:20,3,31.54,B
...,...,...,...,...,...,...
495,112,35,2014-01-01 08:56:15,5,55.42,A
496,111,35,2014-01-01 08:57:20,3,34.87,B
497,276,35,2014-01-01 08:58:25,1,10.94,B
498,266,35,2014-01-01 08:59:30,5,19.86,A


In [77]:
sesions_trans_prod = transactions_products.merge(right=sesions_with_zip, on = 'session_id', how = 'inner')
sesions_trans_prod

,transaction_id,session_id,transaction_time,product_id,amount,brand,customer_id,device,session_start,zip_code
0,298,1,2014-01-01 00:00:00,5,127.64,A,2,desktop,2014-01-01 00:00:00,13244
1,2,1,2014-01-01 00:01:05,2,109.48,B,2,desktop,2014-01-01 00:00:00,13244
2,308,1,2014-01-01 00:02:10,3,95.06,B,2,desktop,2014-01-01 00:00:00,13244
3,116,1,2014-01-01 00:03:15,4,78.92,B,2,desktop,2014-01-01 00:00:00,13244
4,371,1,2014-01-01 00:04:20,3,31.54,B,2,desktop,2014-01-01 00:00:00,13244
...,...,...,...,...,...,...,...,...,...,...
495,112,35,2014-01-01 08:56:15,5,55.42,A,3,mobile,2014-01-01 08:44:20,13244
496,111,35,2014-01-01 08:57:20,3,34.87,B,3,mobile,2014-01-01 08:44:20,13244
497,276,35,2014-01-01 08:58:25,1,10.94,B,3,mobile,2014-01-01 08:44:20,13244
498,266,35,2014-01-01 08:59:30,5,19.86,A,3,mobile,2014-01-01 08:44:20,13244


In [79]:
x = sesions_trans_prod.groupby(by='zip_code')['brand'].value_counts()
x

zip_code  brand
13244     B        149
          A         37
60091     B        247
          A         67
Name: count, dtype: int64